# Kaggle with LLMs -- Competition Workflow
How to approach NLP/LLM competitions end-to-end.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Kaggle Platform

| Feature | Details |
|---|---|
| GPU quota | 30 hours/week free (T4 / P100) |
| Public LB | 30% of test set |
| Private LB | 70% of test set (revealed at end) |
| Discussion | Solution sharing after competition |

In [ ]:
steps = [
    (1, 'Understand task',     'Read overview, data description, metric'),
    (2, 'EDA',                 'Shape, dtypes, missing, label distribution'),
    (3, 'Baseline',            'TF-IDF + LogReg in < 30 minutes. Know floor.'),
    (4, 'Feature engineering', 'Text length, entity counts, readability'),
    (5, 'Model selection',     'BERT -> DistilBERT -> DeBERTa. Match compute.'),
    (6, 'Cross-validation',    'Stratified K-fold. Monitor CV and LB correlation.'),
    (7, 'Hyperparameter tuning','LR, batch size, max_len, warmup'),
    (8, 'Ensemble',            'Average 3-5 diverse models.'),
    (9, 'Submission',          'Check for leakage. Match distribution.'),
    (10,'Post-competition',    'Read top solutions. Learn what you missed.'),
]
print('LLM COMPETITION WORKFLOW:')
for step, name, detail in steps:
    print(f'  Step {step:2d}: {name}')
    print(f'          {detail}')

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)
n = 1000
lengths = np.concatenate([
    np.random.normal(150,50,600),
    np.random.normal(400,100,400)
]).clip(10,800).astype(int)
labels = np.random.choice([0,1],n,p=[0.6,0.4])
fig,axes = plt.subplots(1,3,figsize=(14,4))
axes[0].hist(lengths,bins=30,color='#6b21a8',alpha=0.8)
axes[0].set_title('Text Length Distribution')
axes[1].bar(['Neg','Pos'],[(labels==0).sum(),(labels==1).sum()],color=['#dc2626','#059669'])
axes[1].set_title('Label Distribution')
axes[2].boxplot([lengths[labels==0],lengths[labels==1]],labels=['Neg','Pos'])
axes[2].set_title('Length by Label')
plt.tight_layout(); plt.show()
print(f'Imbalance: {(labels==0).sum()/(labels==1).sum():.2f}:1')
print(f'Use max_len = {int(np.percentile(lengths,95))} (95th percentile)')

## DeBERTa Competition Solution

```python
# Top model for classification tasks
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold

MODEL = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    # tokenize, train, predict on val
    # store out-of-fold predictions for ensemble
    pass

print(f'CV AUC: {roc_auc_score(train_df.label, oof_preds):.4f}')
```

In [ ]:
tips = [
    ('Use DeBERTa-v3-base', 'Outperforms BERT/RoBERTa on classification. 86M params.'),
    ('Stratified K-fold',   'Preserve label distribution. Critical for imbalanced data.'),
    ('LB probe early',      'Submit baseline day 1. Confirm CV/LB correlation.'),
    ('Pseudo-labelling',    'Add high-confidence test predictions to training.'),
    ('Diverse ensemble',    'DeBERTa + DistilBERT + RoBERTa > three DeBERTas.'),
    ('No LB overfitting',   'Private LB = 70% of test. Overfit public LB = rank drop.'),
    ('max_len = 95th pct',  'Longer = slower with diminishing returns.'),
]
print('TOP KAGGLE LLM TIPS:')
for tip, detail in tips:
    print(f'  {tip:<25}: {detail}')

---
**Your turn:** Find a current NLP competition on kaggle.com/competitions. Run TF-IDF + LogReg baseline. What is your score?